# 合并 LoRA Adapter → 完整 bfloat16 权重 + 推理速查（Notebook 版）

训练完得到 LoRA adapter（~250 MB）后，把它合并回基础模型，得到一个独立的 bf16 完整权重（~65 GB），便于后续推理直接 `from_pretrained` 加载。

本 notebook 还包含一个**推理 sanity check**：在 test.jsonl 前 N 条上跑生成、与 ground-truth 对比并速查 JSON 合法性。

**前置**：训练已完成、`adapter/` 目录已存在；A100 80GB 推荐 device=auto 走 GPU。

In [ ]:
# ============================================================
# 设置 HF 环境变量 + 加载 yaml 配置
# ============================================================
import os, json
from pathlib import Path
import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src' / 'Fine_tuning' / 'configs' / 'train_config.yaml').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('找不到 train_config.yaml')
    REPO_ROOT = REPO_ROOT.parent

with open(REPO_ROOT / 'src' / 'Fine_tuning' / 'configs' / 'train_config.yaml', 'r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

for k, v in (CFG.get('env') or {}).items():
    if v is not None:
        os.environ.setdefault(k, str(v))

print('HF_HOME =', os.environ.get('HF_HOME'))

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = CFG['model']['name_or_path']
adapter_dir = REPO_ROOT / CFG['adapter_dir']
merged_dir  = REPO_ROOT / CFG['merged_dir']
merged_dir.mkdir(parents=True, exist_ok=True)

print('[路径] base_model :', base_model_name)
print('[路径] adapter_dir:', adapter_dir)
print('[路径] merged_dir :', merged_dir)
assert adapter_dir.exists(), f'adapter 目录不存在：{adapter_dir}'

In [ ]:
# ============================================================
# 加载基础模型（bf16 完整精度，不量化；A100 80GB 可装下）
# ============================================================
DTYPE = torch.bfloat16

base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=DTYPE,                                    # transformers 4.45+ 用 dtype 替代 torch_dtype
    device_map='auto',
    trust_remote_code=CFG['model'].get('trust_remote_code', True),
)
print('[加载] base model OK')

In [ ]:
# ============================================================
# 挂 LoRA adapter 并 merge_and_unload
# ============================================================
peft_model = PeftModel.from_pretrained(base, str(adapter_dir))
print('[加载] LoRA adapter OK')

merged_model = peft_model.merge_and_unload()
print('[合并] merge_and_unload OK')

In [ ]:
# ============================================================
# 保存合并后完整权重 + tokenizer
# ============================================================
merged_model.save_pretrained(str(merged_dir), safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(
    str(adapter_dir),
    trust_remote_code=CFG['model'].get('trust_remote_code', True),
)
tokenizer.save_pretrained(str(merged_dir))

print('[完成] 合并权重 →', merged_dir)

## 推理 sanity check

在 test.jsonl 前几条样本上跑生成、与 ground-truth 对比、速查 JSON 合法性。这只是人工 sanity check，不是正式评估指标。

In [ ]:
# ============================================================
# 1. 读取 test.jsonl 前 N 条样本
# ============================================================
N_SAMPLES = 5                                    # 想看几条改这里
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.1                                # 抽取任务低温度

sft_yaml = CFG['sft']
test_path = REPO_ROOT / sft_yaml['data_path'] / sft_yaml.get('test_file', 'test.jsonl')

samples = []
with open(test_path, 'r', encoding='utf-8') as f:
    for line in f:
        if len(samples) >= N_SAMPLES:
            break
        line = line.strip()
        if line:
            samples.append(json.loads(line))
print(f'[加载] {len(samples)} 条样本 ←', test_path)

In [ ]:
# ============================================================
# 2. 逐条生成 + 对比
# ============================================================
# 显式开启 KV cache，否则 generate 会逐 token 重算 attention，慢 5-10x
merged_model.config.use_cache = True
if getattr(merged_model, 'generation_config', None) is not None:
    merged_model.generation_config.use_cache = True

merged_model.eval()
device = next(merged_model.parameters()).device

for i, sample in enumerate(samples, 1):
    msgs = sample['messages']
    prompt_msgs = [m for m in msgs if m['role'] in ('system', 'user')]
    ground_truth = next((m['content'] for m in msgs if m['role'] == 'assistant'), '')

    try:
        text = tokenizer.apply_chat_template(
            prompt_msgs,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,                  # Qwen3-style 模板支持
        )
    except TypeError:
        text = tokenizer.apply_chat_template(
            prompt_msgs, tokenize=False, add_generation_prompt=True,
        )

    inputs = tokenizer([text], return_tensors='pt').to(device)
    with torch.no_grad():
        gen_ids = merged_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=(TEMPERATURE > 0),
            temperature=TEMPERATURE if TEMPERATURE > 0 else 1.0,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    output_ids = gen_ids[0][inputs.input_ids.shape[1]:].tolist()
    pred = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    user_content = next((m['content'] for m in prompt_msgs if m['role'] == 'user'), '')
    print(f'\n{"="*80}')
    print(f'Sample {i}/{len(samples)} | PMID: {sample.get("PMID", "N/A")}')
    print(f'{"-"*80}')
    print(f'[USER (前 200 字)]\n  {user_content[:200]}{"..." if len(user_content) > 200 else ""}')
    print(f'\n[GROUND-TRUTH (前 500 字)]\n  {ground_truth[:500]}{"..." if len(ground_truth) > 500 else ""}')
    print(f'\n[MODEL 输出 (前 500 字)]\n  {pred[:500]}{"..." if len(pred) > 500 else ""}')

    try:
        parsed = json.loads(pred)
        if isinstance(parsed, list):
            print(f'\n[质量速查] JSON 合法 ✅  抽出 {len(parsed)} 条三元组')
        else:
            print(f'\n[质量速查] JSON 合法但顶层不是数组 ⚠️')
    except json.JSONDecodeError as e:
        print(f'\n[质量速查] JSON 解析失败 ❌  {e.msg[:100]}')

print(f'\n{"="*80}\n推理 sanity check 完成\n{"="*80}')